## Nädal 7 grupitöö alaülesande roll D kood

In [ ]:
# Antud failis on näha nädal 7 grupitööst minu alaülesande roll D väljundi kood (terviklik kood nähtav meeskonna notebookis team kaustas: team/week7_rfm_complete.ipynb). 
# # Kuna andmete visualiseerimiseks on kasutatud Plotly teeki, siis GitHubi vaatur ei oska neid kuvada, mistõttu on lisatud diagrammidest eraldi kuvatõmmised individual kausta.

# Ülesandeks on luua 3 Plotly diagrammi RFM tulemuste esitamiseks. Koostada ka lühike äritõlgendus: peamised leiud ja soovitused.

# Impordin Plotly
import plotly.express as px

# Loon tulpdiagrammi, mis näitab iga segmendi klientide arvu
fig1 = px.bar(
    segments,
    x='segment',
    y='total_customers',
    title='UrbanStyle kliendibaasi jaotus segmentide lõikes',
    labels={'segment': 'Kliendisegment', 'total_customers': 'Klientide arv'},
    text='total_customers',
    color_discrete_sequence=['#009B8D']
)

# Viimistlen diagrammi - eemaldan x-telje pealkirja, muudan y-telje pealkirja, muudan diagrammi tausta valgeks
fig1.update_layout(
    xaxis={'categoryorder':'total descending'},
    xaxis_title='', 
    yaxis_title='Klientide arv',
    plot_bgcolor='white'
)

fig1.show()



# Loon RFM hajuvusdiagrammi, mis näitab kliendisuhete dünaamikat: Recency vs Monetary

# Loon kõigepealt uue tabeli rfm_names, kus on olemas ka klientide ees- ja perekonnanimed
rfm_names = pd.merge(
    rfm,
    df_customers[['customer_id', 'first_name', 'last_name']],
    on='customer_id',
    how='left'
)

# Loon uue veeru, mis ühendab ees- ja perekonnanime
rfm_names['full_name'] = rfm_names['first_name'] + ' ' + rfm_names['last_name']

fig2 = px.scatter(
    rfm_names,
    x='days_since_last_purchase', 
    y='total_spending', 
    color='segment', 
    size='total_purchases',
    hover_name='full_name',
    title='UrbanStyle RFM-analüüs: VIP-kliendid toovad suurima tulu ja on aktiivseimad',
    labels={
        'full_name': 'Kliendi nimi',
        'days_since_last_purchase': 'Päevi viimasest ostust', 
        'total_spending': 'Kogukulutus (€)',
        'segment': 'Segment',
        'total_purchases': 'Ostude arv'
    },
    hover_data={
        'full_name': False,
        'customer_id': True,
        'days_since_last_purchase': True,
        'total_spending': ':.2f€'
    }
)

# Muudan diagrammi tausta valgeks
fig2.update_layout(
    plot_bgcolor='white'
)

# Lisan keskmise kulutuse viitejoone
# Arvutan esmalt keskmise kulutuse
avg_spending = rfm['total_spending'].mean()

fig2.add_hline(
    y=avg_spending, 
    line_dash='dot', 
    line_color='#E5E7EB', 
    annotation_text=f'Keskmine: {avg_spending:.0f}€', 
    annotation_position='top right'
)

# Lisan VIP Champions segmendi annotatsiooni
fig2.add_annotation(
    x=20,
    y=rfm['total_spending'].max() * 1.1,
    text="<b>VIP Champions: väärtuslikum segment</b>",
    showarrow=True,
    arrowhead=2,
    ax=60,
    ay=-80,
    bgcolor="white"
)

fig2.show()



# Loon tulpdiagrammi, mis näitab TOP 10 VIP klienti kogukulutuse järgi
top_10_vips = rfm_names[rfm_names['segment'] == 'VIP Champions'].nlargest(10, 'total_spending')

# Muudan kliendi-ID väärtuse esmalt täisarvuks ja siis tekstiks, et Plotly ei prooviks seda diagrammil uuesti numbrilise skaalana tõlgendada, vaid käsitleks seda diskreetse sildina
top_10_vips['customer_id'] = top_10_vips['customer_id'].astype(int).astype(str)
top_10_vips['total_spending'] = top_10_vips['total_spending'].round()

fig3 = px.bar(
    top_10_vips,
    x='total_spending',
    y='full_name',
    orientation='h',
    title='UrbanStyle TOP 10 VIP klienti kogukulutuse järgi',
    labels={
        'total_spending': 'Kogukulutus (€)',
        'full_name': 'Kliendi nimi',
        'customer_id': 'Kliendi ID'
    },
    color_discrete_sequence=['#009B8D'],
    text='total_spending',
    hover_data={
        'customer_id': True,
        'total_spending': ':.0f€',
        'full_name': False
    },
)

# Järjestan tulbad suuruse järgi, teen diagrammi tausta valgeks
fig3.update_layout(
    yaxis={'categoryorder':'total ascending'},
    plot_bgcolor='white'
)

fig3.show()



# Leian kõikide segmentide osakaalu kogukäibest
# 1. Grupeerin segmendi järgi ja liidan kokku nende kulutused
segmentide_analuus = segments.groupby('segment')['total_spending'].sum().reset_index()

# 2. Leian terve ettevõtte kogukäibe (kõikide segmentide summa)
ettevotte_kogukäive = segmentide_analuus['total_spending'].sum()

# 3. Arvutan iga segmendi protsendilise osakaalu
segmentide_analuus['osakaal_%'] = (segmentide_analuus['total_spending'] / ettevotte_kogukäive) * 100

# 4. Sorteerin tulemuse kahanevalt, et kõige väärtuslikumad grupid oleksid eespool
segmentide_analuus = segmentide_analuus.sort_values(by='osakaal_%', ascending=False)

# Kuvan tulemuse ümardatuna (nii on mugavam lugeda)
print("UrbanStyle'i segmentide käibeosakaalud:")
print(segmentide_analuus[['segment', 'total_spending', 'osakaal_%']].round(1))



# Loon markdown lahtri, kuhu lisan äritõlgenduse ja soovitused.


KeyboardInterrupt



### Äritõlgendus:
1. ***VIP Champions* (453 klienti):** Tuvastatud on 453 tippklienti, kes on UrbanStyle'i äri kõige olulisemad kliendid. Just see väike, kuid lojaalne grupp annab ligi 43% meie kogukäibest. Nad ostavad sagedasti, kulutavad keskmisest oluliselt rohkem ja on teinud oma viimase ostu hiljuti.  

2. ***At-Risk* kliendid (525 klienti):** Tuvastasime 525 "At-Risk" klienti, mis on kriitiline ohumärk. Tegemist on klientidega, kes olid varem väga väärtuslikud ja sagedased ostjad, kuid pole nüüdseks juba kuude kaupa ühtegi tehingut teinud. Kuna see segment on arvuliselt suurem kui meie VIP-ide rühm, on potentsiaalne käibekadu märkimisväärne, kui me kohe ei sekku.  

3. ***Potential* kliendid (768 klienti):** See on meie suurim segment ja ühtlasi suurim kasvuvõimalus. Need kliendid on hiljuti ostnud ja näitavad positiivset huvi, kuid nad vajavad täiendavat "tõuget", et muutuda püsiklientideks.

<br>

### Soovitused:
Kuna segmenteeritud kampaaniad on kuni kolm korda efektiivsemad kui üldised pakkumised, soovitame turundusjuhile järgmisi samme:
* **VIP programm (eritingimused *VIP Champions* segmendile):**
   - Fookus peab olema eksklusiivsusel ja tunnustusel, mitte massilistel allahindlustel.
   - Pakume neile varajast ligipääsu uutele kollektsioonidele, personaalseid tänusõnumeid ja alati tasuta tarnet, et premeerida nende lojaalsust ilma marginaali liigselt kahjustamata.
* **Win-back kampaania (*At-Risk* klientidele):**
   - Eesmärk on võita nad tagasi enne, kui nad muutuvad lõplikult "kadunud" klientideks.
   - Saadame neile personaalse e-kirja sõnumiga "Me igatseme teid!", mis sisaldab tootesoovitusi nende varasema ostuajaloo põhjal ja ajaliselt piiratud (7 päeva) 15% sooduskoodi, et tekitada kiireloomulisust.
* **Uute klientide kaasamine (*Nurture* programm *Potential* segmendile):**
   - Kasutame lojaalsuspunktide süsteemi, kus iga ost viib kliendi järgmisele tasemele.
   - Rakendame suunatud sõnumeid nagu "Veel üks ost ja sa oled VIP!", et julgustada neid looma UrbanStyle'ist ostmise harjumust ja tõsta nende ostusagedust.

Kirjeldatud andmepõhine lähenemine asendab senise "kõhutunde" strateegilise juhtimisega, võimaldades meil optimeerida turunduseelarvet ja maksimeerida klientide lojaalsust ja ostusagedust.

<br>

### Puuduvad andmed:
* Tagastatud tellimuste info